# Document OCR Pipeline — Quantization (GPTQ, AWQ, SmoothQuant, SpinQuant, ConvRot)

*Part 2 of 4 · Shrinking $f_\theta$ without breaking OCR*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant.ipynb

---

## Why quantize?

A linear layer stores weight matrix $\mathbf{W} \in \mathbb{R}^{O \times I}$. In fp16 that is $2OI$ bytes. With $b$-bit symmetric quantization:

$$
Q(\mathbf{W}) = \text{round}\!\left(\frac{\mathbf{W}}{s}\right), \quad s = \frac{\max |\mathbf{W}|}{2^{b-1}-1}, \quad
\hat{\mathbf{W}} = Q(\mathbf{W}) \cdot s
$$

For $b=4$, storage drops $\approx 4\times$ vs fp16 (plus small scale overhead). The hard part is choosing $Q$ so $\|\mathbf{W} - \hat{\mathbf{W}}\|$ stays small **in the directions calibration data actually uses**.

This notebook pairs with [01 — Document OCR Pipeline](01_document_ocr_pipeline.ipynb). Same Florence-2 OCR setup — but here we shrink the model the right way.

Everything is written **by hand in PyTorch**. No `auto-gptq`, no `awq`, no `bitsandbytes`, no `optimum-quanto`.

| Stage | What you learn |
|-------|----------------|
| **Building blocks** | int4/int8 quantize + drop-in `QuantizedLinear` |
| **GPTQ / AWQ / SmoothQuant / SpinQuant / ConvRot** | Five algorithms, implemented from scratch |
| **SmoothQuant demo** | Alpha sweep + act-range before/after on a real layer |
| **SpinQuant demo** | Learned Givens rotations + output MSE vs RTN |
| **ConvRot demo** | Group-wise RHT (regular Hadamard) — O(K) rotation |
| **5-method OCR compare** | GPTQ · AWQ · SmoothQuant · SpinQuant · ConvRot |
| **Layer inventory** | Which `Linear` layers exist, how big they are, which are worth touching |
| **Sensitivity scan** | Which layers break when you quantize them (output error, not just weight error) |
| **Mixed precision plan** | int4 on safe layers, int8 on medium, fp16 on sensitive ones |
| **Apply + verify** | Swap layers in, run OCR, check the overlay |

Pick **`QUANT_METHOD`** in config. Turn on **`MIXED_PRECISION`** for the full workflow. GPU recommended.

**Series:** [01 OCR](01_document_ocr_pipeline.ipynb) → **02 Quant** → [03 Mobile export](03_ocr_pipeline_mobile.ipynb) → [04 Mobile complete](04_ocr_pipeline_mobile_complete.ipynb)

## 0 — Install & config

Quantization is **post-training**: we freeze $\theta$ and replace selected $\mathbf{W}$ with $\hat{\mathbf{W}}$. Calibration data $\{\mathbf{X}^{(m)}\}_{m=1}^M$ from real OCR forwards supplies the statistics each method needs.

**Steps:**

1. Set **`QUANT_METHOD`** (`gptq` | `awq` | `smoothquant` | `spinquant` | `convrot`) — or use **Stage 8** to compare all five.
2. Set **`MIXED_PRECISION = True`** to auto-pick int4 / int8 / fp16 per layer (recommended).
3. Run the install cell. If Colab restarts, click **Run all**.

**Plain deps only** — no quant libraries. That forces us to implement the math, not hide it behind APIs.

In [ ]:
import os
import re
import subprocess
import sys

# ── CONFIG — change these before you run ─────────────────
QUANT_METHOD = "gptq"          # gptq | awq | smoothquant | spinquant | convrot
MIXED_PRECISION = True         # True = int4/int8/fp16 per layer; False = one BITS everywhere
BITS = 4                       # used only when MIXED_PRECISION = False
MAX_CALIB_BATCHES = 8          # OCR passes for calibration + sensitivity
MAX_QUANT_LAYERS = None        # None = all Linear layers; or cap for quick tests
MAX_ANALYZE_LAYERS = None      # None = analyze all; lower number for faster Colab runs
TASK = "detect"                # detect | ocr — sanity check after quant
MODEL_ID = "microsoft/Florence-2-base-ft"

# Mixed-precision policy (percentiles of sensitivity ranking)
FP16_SENSITIVE_PCT = 15        # top 15% most sensitive → keep fp16
INT8_MID_PCT = 35              # next 35% → int8; rest → int4
MIN_PARAMS_TO_QUANT = 4096     # skip tiny layers (not worth quantizing)
ALWAYS_FP16_PATTERNS = (       # never quantize layers whose name contains:
    "lm_head", "embed", "pooler", "classifier",
    "visual_projection", "image_projection", "vision_tower",
)

SMOOTHQUANT_ALPHA = 0.5
SPINQUANT_REFINE_STEPS = 20    # Cayley/Givens refinement steps on calibration data
SPINQUANT_LR = 0.05
SPINQUANT_N_GIVENS = 48        # learned Givens rotations (SpinQuant from scratch)
SPINQUANT_MAX_DIM = 1024       # skip full learn if in_features larger (use fewer Givens)
CONVROT_GROUP_SIZE = 256       # RHT block N_0 — power of 4: 16, 64, 256, 1024
AWQ_GRID_STEPS = 20
GPTQ_BLOCK_SIZE = 128
GPTQ_DAMPING = 0.01
RUN_ALL_METHODS_OCR = True   # Stage 8: compare all five quant methods
COMPARE_LAYER_MSE = True     # Bonus: weight+output MSE on one layer for all 5 methods

def _pip_version(package):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "show", package],
        capture_output=True, text=True, check=False,
    )
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""

def _restart_runtime():
    print("Restarting runtime so the correct transformers version loads...")
    os.kill(os.getpid(), 9)

def ensure_transformers():
    target = "4.49.0"
    ok = lambda v: v.startswith("4.49")
    pip_ver = _pip_version("transformers")
    if not ok(pip_ver):
        print(f"Installing transformers {target} (pip had {pip_ver or 'none'})...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "transformers==4.49.0",
        ])
        pip_ver = _pip_version("transformers")
        if not ok(pip_ver):
            raise RuntimeError(f"Could not install transformers {target}")
    try:
        import transformers
        loaded = transformers.__version__
    except ImportError:
        loaded = None
    if loaded and not ok(loaded):
        print(f"pip has {pip_ver} but Python still has {loaded}. Restarting...")
        _restart_runtime()
    return pip_ver

# Plain deps only — no quant libraries
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "scipy>=1.12", "scikit-learn",
    "torch", "accelerate", "pillow", "matplotlib", "requests", "huggingface_hub",
])

_tf_ver = ensure_transformers()
print(f"Ready — {QUANT_METHOD.upper()}  |  mixed_precision={MIXED_PRECISION}  |  transformers {_tf_ver}")
print("All quant code is plain PyTorch — no auto-gptq, awq, bitsandbytes, or quanto.")

---
## Stage 1 — Building blocks

Shared primitives for all three quantization methods.

### Symmetric quantization

For bit width $b$, quantile grid $\mathcal{Q}_b = \{-2^{b-1}, \ldots, 2^{b-1}-1\}$:

$$
s = \frac{\max |w|}{2^{b-1}-1}, \quad q = \Pi_{\mathcal{Q}_b}\!\left(\text{round}\!\left(\frac{w}{s}\right)\right), \quad \hat{w} = q \cdot s
$$

where $\Pi_{\mathcal{Q}_b}$ clips to the nearest representable integer.

**Theorem (uniform $L_\infty$ bound):** for non-clipped $w$ (i.e. $|w/s| \le 2^{b-1}-1$):

$$
|w - \hat{w}| = \left|w - s \cdot \text{round}\!\left(\frac{w}{s}\right)\right| \le \frac{s}{2} = \frac{\max|w|}{2(2^{b-1}-1)}
$$

**Proof:** write $w/s = n + \epsilon$ with $n \in \mathbb{Z}$, $|\epsilon| \le 1/2$. Then $|w - s\cdot\text{round}(w/s)| = s|\epsilon| \le s/2$.

**Quantization noise variance** (rough): $\mathbb{E}[(w-\hat{w})^2] \approx s^2/12$ under uniform $\epsilon$ (same as uniform mid-tread quantizer).

### `QuantizedLinear`

Forward paths:
- **GPTQ / AWQ:** $\mathbf{y} = \mathbf{x} \cdot \text{dequant}(Q(\mathbf{W}))^\top$
- **SmoothQuant:** $\mathbf{y} = (\mathbf{x} \odot s) \cdot \text{dequant}(Q(\mathbf{W}/s))^\top$
- **SpinQuant:** $\mathbf{y} = (\mathbf{x}\mathbf{R}) \cdot \text{dequant}(Q(\mathbf{W}\mathbf{R}))^\top$
- **ConvRot:** $\mathbf{y} = \sum_i \text{RHT}(\mathbf{x}_i)\, \text{RHT}(\mathbf{W}_i)^\top$ (group size $N_0$)

Notebook 03 replaces fake quant with integer GEMM.

In [ ]:
from __future__ import annotations

import math
import re
import time
from dataclasses import dataclass
from io import BytesIO
from typing import Callable

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")


def qmax_for_bits(n_bits: int) -> int:
    return 2 ** (n_bits - 1) - 1


def symmetric_quantize_per_channel(W: torch.Tensor, n_bits: int = 4):
    """W: [out, in] → int8 weights + one scale per output channel."""
    qmax = qmax_for_bits(n_bits)
    W = W.float()
    max_abs = W.abs().amax(dim=1).clamp(min=1e-8)
    scales = max_abs / qmax
    q = torch.round(W / scales.unsqueeze(1)).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scales


def symmetric_dequant_per_channel(q: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    return q.float() * scales.unsqueeze(1)


@dataclass
class QuantizedLinearState:
    weight_q: torch.Tensor
    weight_scales: torch.Tensor
    bias: torch.Tensor | None
    act_scales: torch.Tensor | None = None   # SmoothQuant input scales
    givens_angles: torch.Tensor | None = None   # SpinQuant: [K] rotation angles
    givens_pairs: torch.Tensor | None = None      # SpinQuant: [K, 2] index pairs
    convrot_group_size: int | None = None         # ConvRot: RHT block size N_0 (power of 4)
    convrot_pad: int = 0                        # ConvRot: input-dim padding cols
    method: str = "none"
    n_bits: int = 4


def givens_rotation_matrix(angles: torch.Tensor, pairs: torch.Tensor, n: int, device, dtype=torch.float32):
    """Build R = G_K ... G_1 from learned Givens angles (SpinQuant / QuaRot style)."""
    R = torch.eye(n, device=device, dtype=dtype)
    for k in range(angles.shape[0]):
        i, j = int(pairs[k, 0]), int(pairs[k, 1])
        if i == j:
            continue
        c = torch.cos(angles[k])
        s = torch.sin(angles[k])
        Gi = torch.eye(n, device=device, dtype=dtype)
        Gi[i, i] = c
        Gi[j, j] = c
        Gi[i, j] = -s
        Gi[j, i] = s
        R = Gi @ R
    return R


# ── ConvRot: group-wise Regular Hadamard Transform (RHT) ──
_H4_BASE = torch.tensor(
    [[1, 1, 1, -1], [1, 1, -1, 1], [1, -1, 1, 1], [-1, 1, 1, 1]], dtype=torch.float32
)


def regular_hadamard_matrix(n: int, device, dtype=torch.float32) -> torch.Tensor:
    """Regular H-matrix of order n=4^k via Kronecker product (ConvRot paper Eq. 9)."""
    if n < 4:
        raise ValueError(f"ConvRot group size must be >= 4, got {n}")
    t = n
    while t % 4 == 0:
        t //= 4
    if t != 1:
        raise ValueError(f"ConvRot group size must be power of 4, got {n}")
    H = _H4_BASE.to(device=device, dtype=dtype)
    while H.shape[0] < n:
        H = torch.kron(H, _H4_BASE.to(device=device, dtype=dtype))
    H = H[:n, :n] / math.sqrt(n)
    return H


def convrot_rotation_matrix(in_features: int, group_size: int, device, dtype=torch.float32):
    """Block-diagonal RHT: O(K) vs global O(K^2). Returns (R, pad_cols)."""
    pad = (group_size - in_features % group_size) % group_size
    K = in_features + pad
    H = regular_hadamard_matrix(group_size, device, dtype)
    R = torch.block_diag(*([H] * (K // group_size)))
    return R, pad


def apply_convrot_rht(x: torch.Tensor, group_size: int, pad: int) -> torch.Tensor:
    """Apply group-wise RHT to activations [..., in_features]."""
    if pad:
        x = F.pad(x, (0, pad))
    R, _ = convrot_rotation_matrix(x.shape[-1], group_size, x.device, x.dtype)
    return x @ R


class QuantizedLinear(nn.Module):
    """Drop-in Linear with int weights; SmoothQuant / SpinQuant adjust forward path."""

    def __init__(self, in_features: int, out_features: int, state: QuantizedLinearState):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.register_buffer("act_scales", state.act_scales)
        if state.givens_angles is not None:
            self.register_buffer("givens_angles", state.givens_angles)
            self.register_buffer("givens_pairs", state.givens_pairs)
        else:
            self.givens_angles = None
            self.givens_pairs = None
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None
        self.method = state.method
        self.n_bits = state.n_bits
        self.convrot_group_size = state.convrot_group_size
        self.convrot_pad = state.convrot_pad or 0

    def _rotation(self, device, dtype):
        if self.givens_angles is None:
            return None
        return givens_rotation_matrix(
            self.givens_angles.to(device), self.givens_pairs.to(device),
            self.in_features, device, dtype,
        )

    @property
    def weight_fp(self) -> torch.Tensor:
        """Reconstructed fp weight. SpinQuant: W = dequant(W') @ R^T. ConvRot: same with RHT."""
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales)
        if self.act_scales is not None:
            w = w / self.act_scales.unsqueeze(0)
        if self.convrot_group_size is not None:
            if self.convrot_pad:
                w = F.pad(w, (0, self.convrot_pad))
            R, _ = convrot_rotation_matrix(w.shape[1], self.convrot_group_size, w.device, w.dtype)
            w = w @ R.T
            if self.convrot_pad:
                w = w[:, : self.in_features]
            return w
        R = self._rotation(w.device, w.dtype)
        if R is not None:
            w = w @ R.T
        return w

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.convrot_group_size is not None:
            x = apply_convrot_rht(x, self.convrot_group_size, self.convrot_pad)
        else:
            R = self._rotation(x.device, x.dtype)
            if R is not None:
                x = x @ R
        if self.act_scales is not None:
            x = x * self.act_scales.to(x.dtype)
        w = symmetric_dequant_per_channel(self.weight_q, self.weight_scales).to(x.dtype)
        y = F.linear(x, w, self.bias)
        return y

    def storage_bytes(self) -> int:
        n = self.weight_q.numel() + self.weight_scales.numel()
        if self.bias is not None:
            n += self.bias.numel()
        if self.act_scales is not None:
            n += self.act_scales.numel()
        if self.givens_angles is not None:
            n += self.givens_angles.numel() + self.givens_pairs.numel()
        return n * 4  # scales/bias in fp32; q stored as int8


print("Building blocks loaded — ready for GPTQ / AWQ / SmoothQuant / SpinQuant / ConvRot.")


def layer_weight_mse(orig: nn.Linear, quant: QuantizedLinear) -> float:
    with torch.no_grad():
        err = (orig.weight.float() - quant.weight_fp).pow(2).mean().item()
    return err


def layer_output_mse(orig: nn.Linear, quant: QuantizedLinear, inputs: torch.Tensor | None) -> float:
    """Output MSE through the layer — uses quant.forward (SmoothQuant / SpinQuant aware)."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    x = inputs[:2048].to(orig.weight.device)
    with torch.no_grad():
        y_fp = F.linear(x, orig.weight.float(), orig.bias)
        y_q = quant(x)
        return (y_fp - y_q).pow(2).mean().item()

---
## Stage 2 — GPTQ

Paper: *GPTQ* (Frantar et al., 2023).

### Objective

For calibration inputs $\mathbf{X} \in \mathbb{R}^{M \times I}$:

$$
\mathcal{L}(\mathbf{W}) = \|\mathbf{X}\mathbf{W}^\top - \mathbf{X}\hat{\mathbf{W}}^\top\|_F^2 = \text{tr}\bigl((\mathbf{W}-\hat{\mathbf{W}})\mathbf{H}(\mathbf{W}-\hat{\mathbf{W}})^\top\bigr)
$$

with **Hessian** $\mathbf{H} = 2\mathbf{X}^\top\mathbf{X} \in \mathbb{R}^{I \times I}$ (positive semi-definite).

### Full column-update derivation

Quantize column $i$ to $\hat{w}_i$, error $\delta_i = w_i - \hat{w}_i$. Under local quadratic model, optimal compensation for unquantized columns $j > i$:

$$
\Delta w_j = -\frac{\delta_i}{\mathbf{H}_{ii}} \mathbf{H}_{ij}, \quad j > i
$$

**Proof:** set $\partial \mathcal{L}/\partial w_j = 0$ for free variables $w_j$, holding $\hat{w}_i$ fixed. With $\mathcal{L} = (w_i - \hat{w}_i)^2 \mathbf{H}_{ii} + 2\sum_{j>i}(w_i - \hat{w}_i)\mathbf{H}_{ij}(w_j - \hat{w}_j) + \cdots$, the optimal update for column block $j$ is the Schur complement step — equivalent to one step of greedy Cholesky on $\mathbf{H}^{-1}$.

**Implementation:** precompute $\mathbf{H}^{-1}$ via Cholesky on $(\mathbf{H} + \lambda \mathbf{I})$; damp $\lambda > 0$ when $\mathbf{H}_{ii} \to 0$ (dead channels).

In [ ]:
class GPTQQuantizer:
    """GPTQ for one nn.Linear — Hessian from calibration, column-by-column quant."""

    def __init__(
        self,
        layer: nn.Linear,
        n_bits: int = 4,
        block_size: int = 128,
        damping: float = 0.01,
    ):
        self.layer = layer
        self.n_bits = n_bits
        self.block_size = block_size
        self.damping = damping
        self.H: torch.Tensor | None = None
        self.nsamples = 0

    def add_batch(self, inp: torch.Tensor):
        """inp: [batch, seq, in] or [batch, in]."""
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])
        inp = inp.float()
        if self.H is None:
            self.H = torch.zeros((inp.shape[1], inp.shape[1]), device=inp.device)
        batch = inp.shape[0]
        self.H += inp.t() @ inp
        self.nsamples += batch

    def quantize(self) -> QuantizedLinearState:
        W = self.layer.weight.data.float().clone()
        H = self.H.clone()
        dead = torch.diag(H) == 0
        H[dead, dead] = 1.0
        W[:, dead] = 0.0

        damp = self.damping * torch.mean(torch.diag(H))
        diag_idx = torch.arange(H.shape[0], device=H.device)
        H[diag_idx, diag_idx] += damp

        H = torch.linalg.cholesky(H)
        Hinv = torch.cholesky_inverse(H)
        Hinv = torch.linalg.cholesky(Hinv, upper=True)

        Q = torch.zeros_like(W)
        qmax = qmax_for_bits(self.n_bits)

        for i1 in range(0, W.shape[1], self.block_size):
            i2 = min(i1 + self.block_size, W.shape[1])
            count = i2 - i1
            W1 = W[:, i1:i2].clone()
            Q1 = torch.zeros_like(W1)
            Err1 = torch.zeros_like(W1)
            Hinv1 = Hinv[i1:i2, i1:i2]

            for i in range(count):
                w = W1[:, i]
                d = Hinv1[i, i]
                max_abs = w.abs().max().clamp(min=1e-8)
                scale = max_abs / qmax
                q = torch.round(w / scale).clamp(-qmax - 1, qmax)
                Q1[:, i] = q
                err = (w - q * scale) / d
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
                Err1[:, i] = err

            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]

        # Recompute per-channel scales from final Q (for storage / inference)
        weight_q, weight_scales = symmetric_quantize_per_channel(Q, self.n_bits)
        return QuantizedLinearState(
            weight_q=weight_q.cpu(),
            weight_scales=weight_scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            method="gptq",
            n_bits=self.n_bits,
        )


print("GPTQ quantizer ready.")

---
## Stage 3 — AWQ

Paper: *AWQ* (Lin et al., 2023).

### Weighted output error

For one output element $y = \sum_j x_j w_j$, quantization error:

$$
\Delta y = \sum_j x_j (w_j - \hat{w}_j)
$$

Under independence approximation:

$$
\mathbb{E}[\Delta y^2] \approx \sum_j \mathbb{E}[x_j^2] \cdot \mathbb{E}[(w_j - \hat{w}_j)^2]
$$

**Proof (variance propagation):** if $\text{Cov}(x_j, w_j - \hat{w}_j) = 0$ and channels independent, $\text{Var}(\sum_j x_j \epsilon_j) = \sum_j \mathbb{E}[x_j^2]\mathbb{E}[\epsilon_j^2]$ where $\epsilon_j = w_j - \hat{w}_j$.

### Per-channel scaling

Search $s_j > 0$ with $\hat{w}_j = Q(w_j / s_j) \cdot s_j$. Grid over $r \in [0,1]$:

$$
s_j(r) \propto \left(\max_m |X_j^{(m)}|\right)^r
$$

Pick $r^\star = \arg\min_r \sum_m \|\mathbf{X}^{(m)}\mathbf{W}^\top - \mathbf{X}^{(m)}\hat{\mathbf{W}}(r)^\top\|_2^2$.

**Intuition proof:** large $|x_j|$ inflates $\mathbb{E}[x_j^2]$; increasing $s_j$ shrinks $w_j/s_j$ before rounding → smaller effective output error from channel $j$.

In [ ]:
class AWQQuantizer:
    """AWQ for one nn.Linear — scale search using activation magnitudes."""

    def __init__(self, layer: nn.Linear, n_bits: int = 4, grid_steps: int = 20):
        self.layer = layer
        self.n_bits = n_bits
        self.grid_steps = grid_steps
        self.act_scales: torch.Tensor | None = None

    def add_batch(self, inp: torch.Tensor):
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])
        batch_scale = inp.abs().amax(dim=0).float()
        if self.act_scales is None:
            self.act_scales = batch_scale
        else:
            self.act_scales = torch.maximum(self.act_scales, batch_scale)

    def quantize(self) -> QuantizedLinearState:
        W = self.layer.weight.data.float()
        act = self.act_scales.to(W.device).clamp(min=1e-6)

        best_error = float("inf")
        best_q, best_scales = None, None

        for step in range(self.grid_steps):
            ratio = step / self.grid_steps
            s = act.pow(ratio).clamp(min=1e-6)
            W_scaled = W * s.unsqueeze(0)
            q, ch_scales = symmetric_quantize_per_channel(W_scaled, self.n_bits)
            W_hat = symmetric_dequant_per_channel(q, ch_scales) / s.unsqueeze(0)
            err = ((W - W_hat) * act.unsqueeze(0)).pow(2).sum().item()
            if err < best_error:
                best_error = err
                best_q, best_scales = q, ch_scales

        return QuantizedLinearState(
            weight_q=best_q.cpu(),
            weight_scales=best_scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            method="awq",
            n_bits=self.n_bits,
        )


print("AWQ quantizer ready.")

---
## Stage 4 — SmoothQuant

Paper: *SmoothQuant* (Xiao et al., 2023).

### Outlier problem (formal)

Define channel dynamic range $R_j = \max_m |X_j^{(m)}| / \mathbb{E}_m[|X_j^{(m)}|]$. Outlier channels have $R_j \gg 1$, forcing int8 scale $s_j^X \propto R_j$ and wasting precision on other channels.

### Migration theorem

For $\alpha \in [0,1]$, per-channel:

$$
s_j = \frac{R_j^\alpha}{\|W_j\|_\infty^{1-\alpha}}, \quad W'_j = \frac{W_j}{s_j}, \quad X'_j = X_j \cdot s_j
$$

**Theorem (exact equivalence):** for all $j$:

$$
X_j W_j = X'_j W'_j = (X_j s_j)(W_j / s_j)
$$

**Proof:** direct algebra; summing over $j$: $\mathbf{x}^\top \mathbf{w} = \mathbf{x}'^\top \mathbf{w}'$ for any row vector. Matmul $\mathbf{X}\mathbf{W}^\top$ unchanged column-wise.

**Effect on ranges:** $|X'_j|_{\max} \approx R_j^{1-\alpha}$ and $|W'_j|_{\max} \approx \|W_j\|_\infty^\alpha$ — trade activation outliers for weight scale at controllable rate $\alpha$. Quantize $\mathbf{W}'$; fold $s_j$ into runtime.

In [ ]:
class SmoothQuantQuantizer:
    """SmoothQuant for one nn.Linear — balance act/weight ranges, then quantize."""

    def __init__(self, layer: nn.Linear, n_bits: int = 4, alpha: float = 0.5):
        self.layer = layer
        self.n_bits = n_bits
        self.alpha = alpha
        self.act_max: torch.Tensor | None = None

    def add_batch(self, inp: torch.Tensor):
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])
        batch_max = inp.abs().amax(dim=0).float()
        if self.act_max is None:
            self.act_max = batch_max
        else:
            self.act_max = torch.maximum(self.act_max, batch_max)

    def quantize(self) -> QuantizedLinearState:
        W = self.layer.weight.data.float()
        act_max = self.act_max.to(W.device).clamp(min=1e-6)
        weight_max = W.abs().amax(dim=0).clamp(min=1e-6)

        s = act_max.pow(self.alpha) / weight_max.pow(1.0 - self.alpha)
        s = s.clamp(min=1e-6)
        W_smooth = W / s.unsqueeze(0)
        q, ch_scales = symmetric_quantize_per_channel(W_smooth, self.n_bits)

        return QuantizedLinearState(
            weight_q=q.cpu(),
            weight_scales=ch_scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            act_scales=s.cpu(),
            method="smoothquant",
            n_bits=self.n_bits,
        )


print("SmoothQuant quantizer ready.")

---
## Stage 4c — SpinQuant

Paper: *SpinQuant: LLM Quantization with Learned Rotations* (Liu et al., ICLR 2025).

### Rotation invariance

For orthogonal $\mathbf{R}$ ($\mathbf{R}^\top\mathbf{R} = \mathbf{I}$):

$$
\mathbf{Y} = \mathbf{X}\mathbf{W}^\top = (\mathbf{X}\mathbf{R})(\mathbf{W}\mathbf{R})^\top = \mathbf{X}'(\mathbf{W}')^\top
$$

**Proof:** $\mathbf{X}'(\mathbf{W}')^\top = \mathbf{X}\mathbf{R}\mathbf{R}^\top\mathbf{W}^\top = \mathbf{X}\mathbf{W}^\top$ since $\mathbf{R}\mathbf{R}^\top = \mathbf{I}$.

Quantize $\mathbf{W}' = \mathbf{W}\mathbf{R}$ in rotated space where outliers are spread evenly → lower $\text{MSE}_{\text{out}}$ at 4-bit.

### Learned Givens rotations (from scratch)

SpinQuant learns $K$ plane rotations $G_k$:

$$
\mathbf{R} = G_K \cdots G_1, \quad G_k \text{ rotates coordinates } (i_k, j_k) \text{ by angle } \theta_k
$$

We optimize $\{\theta_k\}$ on calibration data:

$$
\min_{\theta} \|\mathbf{X}\mathbf{W}^\top - Q(\mathbf{X}\mathbf{R}_\theta)\, Q(\mathbf{W}\mathbf{R}_\theta)^\top\|_F^2
$$

Then store $(\theta_k, i_k, j_k)$ — compact vs full $\mathbf{R}$ matrix. Stage 4c demo compares SpinQuant vs round-to-nearest on one layer.

In [ ]:
class SpinQuantQuantizer:
    """SpinQuant for one nn.Linear — learn Givens rotations, then quantize W @ R."""

    def __init__(
        self,
        layer: nn.Linear,
        n_bits: int = 4,
        n_givens: int | None = None,
        refine_steps: int | None = None,
        lr: float | None = None,
    ):
        self.layer = layer
        self.n_bits = n_bits
        self.n_givens = n_givens if n_givens is not None else globals().get("SPINQUANT_N_GIVENS", 48)
        self.refine_steps = refine_steps if refine_steps is not None else globals().get("SPINQUANT_REFINE_STEPS", 20)
        self.lr = lr if lr is not None else globals().get("SPINQUANT_LR", 0.05)
        self.calib: list[torch.Tensor] = []

    def add_batch(self, inp: torch.Tensor):
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])
        self.calib.append(inp.detach())

    def _sample_pairs(self, n: int, count: int, device) -> torch.Tensor:
        pairs = []
        for _ in range(count):
            i = torch.randint(0, n, (1,)).item()
            j = torch.randint(0, n, (1,)).item()
            if i == j:
                j = (j + 1) % n
            pairs.append([i, j])
        return torch.tensor(pairs, device=device, dtype=torch.long)

    def _learn_rotation(self, W: torch.Tensor, X: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (angles, pairs) minimizing output MSE after quant in rotated space."""
        _, I = W.shape
        device = W.device
        max_dim = globals().get("SPINQUANT_MAX_DIM", 1024)
        ng = max(8, self.n_givens // 4) if I > max_dim else self.n_givens
        pairs = self._sample_pairs(I, ng, device)
        angles = torch.zeros(ng, device=device, requires_grad=True)
        if X is None or X.numel() == 0:
            return angles.detach().cpu(), pairs.cpu()

        X = X[:2048].to(device).float()
        bias = self.layer.bias
        opt = torch.optim.Adam([angles], lr=self.lr)

        for _ in range(self.refine_steps):
            opt.zero_grad()
            R = givens_rotation_matrix(angles, pairs, I, device)
            W_r = W @ R
            q, scales = symmetric_quantize_per_channel(W_r, self.n_bits)
            W_hat = symmetric_dequant_per_channel(q, scales)
            y_fp = F.linear(X, W, bias)
            y_q = F.linear(X @ R, W_hat, bias)
            loss = (y_fp - y_q).pow(2).mean()
            loss.backward()
            opt.step()

        return angles.detach().cpu(), pairs.cpu()

    def quantize(self) -> QuantizedLinearState:
        W = self.layer.weight.data.float()
        X = torch.cat(self.calib, dim=0) if self.calib else None
        angles, pairs = self._learn_rotation(W, X)
        device = W.device
        R = givens_rotation_matrix(angles.to(device), pairs.to(device), W.shape[1], device)
        W_r = W @ R
        q, scales = symmetric_quantize_per_channel(W_r, self.n_bits)

        return QuantizedLinearState(
            weight_q=q.cpu(),
            weight_scales=scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            givens_angles=angles,
            givens_pairs=pairs,
            method="spinquant",
            n_bits=self.n_bits,
        )


print("SpinQuant quantizer ready.")

---
## Stage 4e — ConvRot

Paper: *ConvRot: Rotation-Based Plug-and-Play 4-bit Quantization* (Huang et al., arXiv 2512.03673).

### Group-wise Regular Hadamard Transform (RHT)

Partition input dimension $K$ into blocks of size $N_0$ (power of 4: 16, 64, 256, 1024):

$$
\mathbf{X} = [\mathbf{X}_1, \ldots, \mathbf{X}_B], \quad \mathbf{W} = [\mathbf{W}_1, \ldots, \mathbf{W}_B], \quad B = \lceil K / N_0 \rceil
$$

Per block, apply regular Hadamard rotation:

$$
\mathbf{Y} = \sum_{i=1}^{B} \text{RHT}(\mathbf{X}_i)\, \text{RHT}(\mathbf{W}_i)^\top
$$

### Regular Hadamard base ($n=4$)

$$
\mathbf{H}_4 = \begin{bmatrix} 1&1&1&-1 \\ 1&1&-1&1 \\ 1&-1&1&1 \\ -1&1&1&1 \end{bmatrix}, \quad
\mathbf{H}_{4^{k+1}} = \mathbf{H}_{4^k} \otimes \mathbf{H}_4
$$

Normalize $\mathbf{H}_n / \sqrt{n}$ so $\mathbf{H}_n \mathbf{H}_n^\top = \mathbf{I}_n$ (**orthogonal**).

**Theorem (column discrepancy):** regular $\mathbf{H}_n$ achieves minimal $\|\mathbf{H}_n^\top \mathbf{1}\|_\infty = \sqrt{n}$, preventing row-wise outlier amplification (vs Sylvester Hadamard).

**Complexity:** global rotation $O(K^2)$ → group-wise ConvRot $O(K)$.

Default $N_0 = 256$ (paper recommendation for accuracy/speed trade-off).

In [ ]:
class ConvRotQuantizer:
    """ConvRot — group-wise RHT then symmetric quantize (no training, plug-and-play)."""

    def __init__(self, layer: nn.Linear, n_bits: int = 4, group_size: int | None = None):
        self.layer = layer
        self.n_bits = n_bits
        self.group_size = group_size if group_size is not None else globals().get("CONVROT_GROUP_SIZE", 256)
        self.calib: list[torch.Tensor] = []

    def add_batch(self, inp: torch.Tensor):
        if inp.dim() == 3:
            inp = inp.reshape(-1, inp.shape[-1])
        self.calib.append(inp.detach())

    def quantize(self) -> QuantizedLinearState:
        W = self.layer.weight.data.float()
        device = W.device
        gs = self.group_size
        n = gs
        while n % 4 == 0:
            n //= 4
        if n != 1 or gs < 4:
            raise ValueError(f"CONVROT_GROUP_SIZE must be power of 4 (16/64/256/1024), got {gs}")

        R, pad = convrot_rotation_matrix(W.shape[1], gs, device)
        W_pad = F.pad(W, (0, pad)) if pad else W
        W_rot = W_pad @ R
        q, scales = symmetric_quantize_per_channel(W_rot, self.n_bits)

        return QuantizedLinearState(
            weight_q=q.cpu(),
            weight_scales=scales.cpu(),
            bias=self.layer.bias.detach().cpu() if self.layer.bias is not None else None,
            convrot_group_size=gs,
            convrot_pad=pad,
            method="convrot",
            n_bits=self.n_bits,
        )


print("ConvRot quantizer ready.")

---
## Stage 4d — SpinQuant demo (RTN vs learned rotation)

Compare **round-to-nearest (RTN)** vs **SpinQuant** output MSE on the same layer and calibration batch:

$$
\text{MSE}_{\text{RTN}} = \|\mathbf{X}\mathbf{W}^\top - Q(\mathbf{X})\, Q(\mathbf{W})^\top\|^2, \quad
\text{MSE}_{\text{Spin}} = \|\mathbf{X}\mathbf{W}^\top - Q(\mathbf{X}\mathbf{R})\, Q(\mathbf{W}\mathbf{R})^\top\|^2
$$

**Expectation:** learned $\mathbf{R}$ reduces $\text{MSE}_{\text{Spin}}$ vs RTN by spreading outlier mass across channels.

In [ ]:
# Re-run this cell AFTER Stage 5 (model load) + calibration helpers below.
print("SpinQuant demo runs after Stage 5 — see cell after activation capture.")

---
## Stage 5 — Load the model & collect calibration data

Hooks on `nn.Linear` accumulate calibration inputs. For layer $\ell$:

$$
\mathbf{H}_\ell = 2 \sum_{m=1}^{M} (\mathbf{X}_\ell^{(m)})^\top \mathbf{X}_\ell^{(m)}
$$

**Sample complexity:** $\mathbf{H}_\ell$ is rank-$M$ if $M < I$; need $M \gtrsim I$ for full rank (rarely achievable). In practice $M \in [32,512]$ suffices because $\mathbf{H}_\ell$ is dominated by top eigenvectors of activation covariance.

**Proof (why real OCR data):** $\mathbf{H}_\ell$ weights input directions by calibration distribution. Random Gaussian $\mathbf{X}$ mis-estimates important directions → GPTQ minimizes wrong objective. Document images + task prompts match production $\mathbf{X}_\ell$ support.

In [ ]:
ensure_transformers()
from transformers import AutoProcessor, AutoModelForCausalLM

PROMPT = "<OCR_WITH_REGION>" if TASK == "detect" else "<OCR>"
dtype = torch.float16 if DEVICE == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=dtype,
    attn_implementation="eager",
).to(DEVICE)
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_ID}")
print(f"{params/1e6:.1f}M params  ·  dtype={dtype}")


def pad_info(image):
    w, h = image.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), "white")
    pad_x, pad_y = (side - w) // 2, (side - h) // 2
    canvas.paste(image, (pad_x, pad_y))
    return canvas, pad_x, pad_y, w, h


def load_sample_image():
    try:
        url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
        return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
    except Exception:
        img = Image.new("RGB", (640, 480), "white")
        ImageDraw.Draw(img).text((20, 20), "Sample document", fill="black")
        return img


image = load_sample_image()
padded, *_ = pad_info(image)
print(f"Using sample page: {image.size[0]}×{image.size[1]} px")

plt.figure(figsize=(6, 4)); plt.imshow(image); plt.title("Document we'll calibrate on"); plt.axis("off"); plt.show()

---
## Stage 4b — SmoothQuant demo (alpha sweep + act ranges)

Stored weights are smoothed: $W'_j = W_j / s_j(\alpha)$. **Undo** before weight MSE:

$$
W_j = W'_j \cdot s_j(\alpha)
$$

### Alpha effect on dynamic range

Define pre-smooth activation max $A_j = \max_m |X_j^{(m)}|$. Post-smooth:

$$
A'_j(\alpha) = A_j \cdot s_j(\alpha) = A_j^{1-\alpha} \cdot \|W_j\|_\infty^\alpha
$$

**Proof:** substitute $s_j = A_j^\alpha / \|W_j\|_\infty^{1-\alpha}$ from SmoothQuant formula.

At $\alpha=0$: $A'_j = A_j$ (no migration). At $\alpha=1$: $A'_j = \|W_j\|_\infty$ (activations fully smoothed). Sweep $\alpha \in \{0.3, 0.5, 0.7\}$ and measure $\text{MSE}_{\text{out}}$ via `quant.forward()`.

In [ ]:
def _layer_prot(name):
    n = name.lower()
    return any(p in n for p in ALWAYS_FP16_PATTERNS)

def _run_cal(m, proc, img, prompt, n):
    padded, *_ = pad_info(img)
    for _ in range(n):
        inputs = proc(text=prompt, images=padded, return_tensors="pt").to(DEVICE)
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(m.parameters()).dtype)
        with torch.no_grad():
            m.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
                       max_new_tokens=64, num_beams=1, use_cache=False)

# Pick first quantifiable Linear layer
_demo_name, demo_layer = next(
    (n, m) for n, m in model.named_modules()
    if isinstance(m, nn.Linear) and not _layer_prot(n) and m.weight.numel() >= MIN_PARAMS_TO_QUANT
)

_demo_cap = {}
_h = demo_layer.register_forward_hook(
    lambda m, inp, out: _demo_cap.setdefault("x", []).append(inp[0].detach().reshape(-1, inp[0].shape[-1]).cpu())
)
_run_cal(model, processor, image, PROMPT, MAX_CALIB_BATCHES)
_h.remove()
demo_inputs = torch.cat(_demo_cap["x"], dim=0) if _demo_cap.get("x") else torch.empty(0)

print(f"SmoothQuant demo layer: {_demo_name}  shape={tuple(demo_layer.weight.shape)}")

alpha_results = []
for alpha in [0.3, 0.5, 0.7]:
    sq = SmoothQuantQuantizer(demo_layer, n_bits=4, alpha=alpha)
    h2 = demo_layer.register_forward_hook(lambda m, inp, out, q=sq: q.add_batch(inp[0].detach()))
    _run_cal(model, processor, image, PROMPT, 2)
    h2.remove()
    state = sq.quantize()
    ql = QuantizedLinear(demo_layer.in_features, demo_layer.out_features, state).to(DEVICE)
    act_before = float(demo_inputs.abs().amax()) if demo_inputs.numel() else 0.0
    act_after = float((demo_inputs * state.act_scales).abs().amax()) if demo_inputs.numel() else 0.0
    alpha_results.append({
        "alpha": alpha,
        "weight_mse": layer_weight_mse(demo_layer, ql),
        "output_mse": layer_output_mse(demo_layer, ql, demo_inputs),
        "act_max_before": act_before,
        "act_max_after": act_after,
    })

print(f"\n{'Alpha':<8} {'Weight MSE':>12} {'Output MSE':>12} {'Act max before':>14} {'Act max after':>14}")
print("-" * 65)
for r in alpha_results:
    print(f"{r['alpha']:<8.1f} {r['weight_mse']:>12.2e} {r['output_mse']:>12.2e} {r['act_max_before']:>14.2f} {r['act_max_after']:>14.2f}")

best = min(alpha_results, key=lambda x: x["output_mse"])
print(f"\nBest alpha for this layer (lowest output MSE): {best['alpha']}  — set SMOOTHQUANT_ALPHA in config")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].bar([str(r["alpha"]) for r in alpha_results], [r["output_mse"] for r in alpha_results], color="#C44E52")
ax[0].set_title("SmoothQuant output MSE by alpha"); ax[0].set_xlabel("alpha")
ax[1].bar(["before", "after"], [alpha_results[1]["act_max_before"], alpha_results[1]["act_max_after"]], color=["#e74c3c", "#27ae60"])
ax[1].set_title("Activation range @ alpha=0.5"); plt.tight_layout(); plt.show()

In [ ]:
@dataclass
class LayerProfile:
    name: str
    shape: tuple
    num_params: int
    pct_of_linear: float
    weight_mse_int4: float
    weight_mse_int8: float
    output_mse_int4: float
    output_mse_int8: float
    act_max: float
    sensitivity: float
    bits: str = "int4"
    protected: bool = False
    note: str = ""


def iter_linear_modules(root: nn.Module, limit: int | None = None):
    out = []
    for name, module in root.named_modules():
        if isinstance(module, nn.Linear):
            out.append((name, module))
    if limit is not None:
        out = out[:limit]
    return out


def layer_matches_patterns(name: str, patterns: tuple) -> bool:
    n = name.lower()
    return any(p.lower() in n for p in patterns)


def build_quantizer(method: str, layer: nn.Linear, n_bits: int | None = None):
    bits = n_bits if n_bits is not None else BITS
    if method == "gptq":
        return GPTQQuantizer(layer, n_bits=bits, block_size=GPTQ_BLOCK_SIZE, damping=GPTQ_DAMPING)
    if method == "awq":
        return AWQQuantizer(layer, n_bits=bits, grid_steps=AWQ_GRID_STEPS)
    if method == "smoothquant":
        return SmoothQuantQuantizer(layer, n_bits=bits, alpha=SMOOTHQUANT_ALPHA)
    if method == "spinquant":
        return SpinQuantQuantizer(
            layer, n_bits=bits, n_givens=SPINQUANT_N_GIVENS,
            refine_steps=SPINQUANT_REFINE_STEPS, lr=SPINQUANT_LR,
        )
    if method == "convrot":
        return ConvRotQuantizer(layer, n_bits=bits, group_size=CONVROT_GROUP_SIZE)
    raise ValueError(f"Unknown QUANT_METHOD: {method}")


def naive_weight_mse(layer: nn.Linear, n_bits: int) -> float:
    q, s = symmetric_quantize_per_channel(layer.weight.data, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s)
    return (layer.weight.float() - w_hat).pow(2).mean().item()


def measure_output_mse(layer: nn.Linear, inputs: torch.Tensor | None, n_bits: int) -> float:
    """How much does quantizing THIS layer shift its output? Key sensitivity signal."""
    if inputs is None or inputs.numel() == 0:
        return float("nan")
    q, s = symmetric_quantize_per_channel(layer.weight.data, n_bits)
    w_hat = symmetric_dequant_per_channel(q, s)
    x = inputs[:2048].to(layer.weight.device)
    with torch.no_grad():
        bias = layer.bias
        y_fp = F.linear(x, layer.weight.float(), bias.float() if bias is not None else None)
        y_q = F.linear(x, w_hat.to(x.dtype), bias)
        return (y_fp - y_q).pow(2).mean().item()


def collect_layer_inputs(model, layer_names, processor, image, prompt, n_batches):
    captures = {n: [] for n in layer_names}
    module_map = dict(model.named_modules())

    def make_hook(name):
        def hook(_module, inp, _out):
            x = inp[0].detach()
            if x.dim() == 3:
                x = x.reshape(-1, x.shape[-1])
            captures[name].append(x.cpu())
        return hook

    handles = [module_map[n].register_forward_hook(make_hook(n)) for n in layer_names]
    run_calibration(model, processor, image, prompt, n_batches)
    for h in handles:
        h.remove()

    for name in layer_names:
        if captures[name]:
            captures[name] = torch.cat(captures[name], dim=0)
        else:
            captures[name] = torch.empty(0)
    return captures


def register_calibration_hooks(model: nn.Module, quantizers: dict[str, object]):
    handles = []

    def make_hook(q):
        def hook(_module, inp, _out):
            q.add_batch(inp[0].detach())
        return hook

    module_map = dict(model.named_modules())
    for name, q in quantizers.items():
        handles.append(module_map[name].register_forward_hook(make_hook(q)))
    return handles


def run_calibration(model, processor, image, prompt, n_batches: int):
    padded, *_ = pad_info(image)
    for _ in range(n_batches):
        inputs = processor(text=prompt, images=padded, return_tensors="pt").to(DEVICE)
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(model.parameters()).dtype)
        with torch.no_grad():
            model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=64,
                num_beams=1,
                use_cache=False,
            )
    print(f"Calibration done — {n_batches} forward passes through the model.")


def build_layer_inventory(all_linears):
    total = sum(m.weight.numel() for _, m in all_linears)
    rows = []
    for name, mod in all_linears:
        n = mod.weight.numel()
        rows.append({
            "name": name,
            "shape": tuple(mod.weight.shape),
            "params": n,
            "pct": 100.0 * n / max(total, 1),
            "protected": layer_matches_patterns(name, ALWAYS_FP16_PATTERNS),
            "tiny": n < MIN_PARAMS_TO_QUANT,
        })
    rows.sort(key=lambda r: r["params"], reverse=True)
    return rows, total


def analyze_sensitivity(all_linears, captures):
    total = sum(m.weight.numel() for _, m in all_linears)
    profiles = []
    for name, mod in all_linears:
        n = mod.weight.numel()
        inp = captures.get(name)
        act_max = float(inp.abs().amax().item()) if inp is not None and inp.numel() else 0.0
        out4 = measure_output_mse(mod, inp, 4)
        out8 = measure_output_mse(mod, inp, 8)
        w4 = naive_weight_mse(mod, 4)
        w8 = naive_weight_mse(mod, 8)
        protected = layer_matches_patterns(name, ALWAYS_FP16_PATTERNS) or n < MIN_PARAMS_TO_QUANT
        # Sensitivity: output error at int4 matters most; activation outliers make it worse
        sens = (out4 if not math.isnan(out4) else w4) * (1.0 + 0.1 * math.log1p(act_max))
        profiles.append(LayerProfile(
            name=name, shape=tuple(mod.weight.shape), num_params=n,
            pct_of_linear=100.0 * n / max(total, 1),
            weight_mse_int4=w4, weight_mse_int8=w8,
            output_mse_int4=out4, output_mse_int8=out8,
            act_max=act_max, sensitivity=sens, protected=protected,
        ))
    profiles.sort(key=lambda p: p.sensitivity, reverse=True)
    return profiles


def assign_mixed_precision(profiles):
    """Rank by sensitivity: most sensitive → fp16, middle → int8, rest → int4."""
    for p in profiles:
        if p.protected:
            p.bits = "fp16"
            p.note = "protected (head/embed/tiny layer)"

    candidates = [p for p in profiles if not p.protected]
    n = len(candidates)
    if n == 0:
        return profiles

    n_fp16 = max(1, round(n * FP16_SENSITIVE_PCT / 100))
    n_int8 = max(0, round(n * INT8_MID_PCT / 100))

    for i, p in enumerate(candidates):
        if i < n_fp16:
            p.bits = "fp16"
            p.note = f"high sensitivity (rank {i+1}/{n}) — keep fp16"
        elif i < n_fp16 + n_int8:
            p.bits = "int8"
            p.note = f"medium sensitivity — int8 is safer"
        else:
            p.bits = "int4"
            p.note = f"low sensitivity — int4 saves most memory"

    return profiles


def replace_linear(model, name, new_module):
    parent_name, _, child_name = name.rpartition(".")
    parent = model.get_submodule(parent_name) if parent_name else model
    setattr(parent, child_name, new_module)


def apply_quant_plan(model, profiles, method, captures=None):
    """Quantize only layers assigned int4/int8; leave fp16 layers untouched."""
    to_quant = [p for p in profiles if p.bits in ("int4", "int8")]
    if not to_quant:
        print("Nothing to quantize — all layers marked fp16.")
        return []

    name_to_mod = dict(iter_linear_modules(model))
    quantizers = {p.name: build_quantizer(method, name_to_mod[p.name], n_bits=int(p.bits.replace("int", "")))
                  for p in to_quant}
    handles = register_calibration_hooks(model, quantizers)
    run_calibration(model, processor, image, PROMPT, MAX_CALIB_BATCHES)

    stats = []
    for p in to_quant:
        mod = name_to_mod[p.name]
        state = quantizers[p.name].quantize()
        qlayer = QuantizedLinear(mod.in_features, mod.out_features, state).to(DEVICE)
        w_mse = layer_weight_mse(mod, qlayer)
        o_mse = layer_output_mse(mod, qlayer, captures.get(p.name) if captures else None)
        replace_linear(model, p.name, qlayer)
        stats.append({
            "layer": p.name, "bits": p.bits, "method": method,
            "weight_mse": w_mse, "output_mse": o_mse,
            "sensitivity": p.sensitivity, "note": p.note,
            "fp_bytes": mod.weight.numel() * 2,
            "q_bytes": qlayer.storage_bytes(),
        })

    for h in handles:
        h.remove()
    return stats


print("Helpers loaded — inventory, sensitivity scan, mixed-precision planner.")

---
## Stage 5b — Which layers should we quantize?

Not every layer is worth touching. Parameter count for Linear $(O,I)$:

$$
N_{\text{params}} = O \cdot I + O \quad (\text{bias included})
$$

**Fraction of total model params:**

$$
f_\ell = \frac{N_{\text{params},\ell}}{\sum_{\ell'} N_{\text{params},\ell'}} \quad \Rightarrow \quad \text{quantizing top-}k\text{ layers by } f_\ell \text{ captures most savings}
$$

Good candidates: large transformer `Linear` blocks. Usually leave alone (fp16): `lm_head`, embed, vision projection; tiny layers ($N_{\text{params}} < 4096$).

The table below lists every `Linear` layer, its size, and whether we auto-protect it.

**Compression:** int4 saves $\approx 4\times$ vs fp16 per layer ($16/b_\ell = 4$ when $b_\ell=4$).

In [ ]:
all_linears = iter_linear_modules(model, MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)
inventory, total_linear_params = build_layer_inventory(all_linears)

print(f"Found {len(inventory)} Linear layers  ·  {total_linear_params/1e6:.2f}M params total\n")
print(f"{'Layer':<55} {'Shape':<18} {'Params':>9} {'%':>6}  Flag")
print("-" * 95)
for row in inventory[:20]:
    flag = "PROTECT" if row["protected"] else ("tiny" if row["tiny"] else "quant?")
    short = row["name"] if len(row["name"]) <= 54 else "…" + row["name"][-53:]
    print(f"{short:<55} {str(row['shape']):<18} {row['params']:>9,} {row['pct']:>5.1f}%  {flag}")
if len(inventory) > 20:
    print(f"... +{len(inventory)-20} more layers")

top5_pct = sum(r["pct"] for r in inventory[:5])
print(f"\nTop 5 layers hold {top5_pct:.0f}% of all Linear params — quantizing these gives the biggest win.")

fig, ax = plt.subplots(figsize=(10, 4))
top = inventory[:15]
ax.barh([r["name"].split(".")[-1] for r in top][::-1], [r["pct"] for r in top][::-1], color="steelblue")
ax.set_xlabel("% of Linear params")
ax.set_title("Biggest layers first — start here when looking for quant targets")
plt.tight_layout(); plt.show()

---
## Stage 5c — Which layers are sensitive?

Weight MSE is a poor proxy. Define **layer output error**:

$$
\text{MSE}_{\text{out}} = \frac{1}{M}\sum_m \|\mathbf{Y}^{(m)} - \hat{\mathbf{Y}}^{(m)}\|_F^2, \quad \mathbf{Y}^{(m)} = \mathbf{X}^{(m)}\mathbf{W}^\top
$$

### Error propagation lemma

If layer $\ell$ output error is $\Delta \mathbf{Y}$, and next layer is linear $\mathbf{Z} = \Delta \mathbf{Y} \mathbf{W}_{\ell+1}^\top$, then:

$$
\|\Delta \mathbf{Z}\|_F \le \|\Delta \mathbf{Y}\|_F \cdot \|\mathbf{W}_{\ell+1}\|_2
$$

**Proof:** submultiplicativity of spectral norm: $\|\Delta \mathbf{Y} \mathbf{W}_{\ell+1}^\top\|_F \le \|\Delta \mathbf{Y}\|_F \|\mathbf{W}_{\ell+1}\|_2$.

So early-layer output error **amplifies** through deep stacks — sensitivity ranking uses:
1. **Output MSE @ int4 / int8**
2. **Outlier score** $R_j = \max |X_j| / \text{mean}(|X_j|)$
3. **Combined score** $S_\ell = w_1 \text{MSE}_{\text{out,int4}} + w_2 \bar{R}_\ell$

Top rank → fp16/int8; bottom → int4.

In [ ]:
layer_names = [n for n, _ in all_linears]
print(f"Capturing activations for {len(layer_names)} layers...")
captures = collect_layer_inputs(model, layer_names, processor, image, PROMPT, MAX_CALIB_BATCHES)

profiles = analyze_sensitivity(all_linears, captures)

print(f"\n{'Rank':<5} {'Sensitivity':>11} {'OutMSE i4':>11} {'OutMSE i8':>11}  Layer")
print("-" * 90)
for i, p in enumerate(profiles[:15], 1):
    tag = " [PROTECT]" if p.protected else ""
    print(f"{i:<5} {p.sensitivity:>11.2e} {p.output_mse_int4:>11.2e} {p.output_mse_int8:>11.2e}  {p.name.split('.')[-1]}{tag}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show = profiles[:min(20, len(profiles))]
labels = [p.name.split(".")[-1] for p in show]
axes[0].barh(labels[::-1], [p.sensitivity for p in show][::-1], color="coral")
axes[0].set_xlabel("Sensitivity score (higher = more fragile)")
axes[0].set_title("Most sensitive layers — quantize these last or use int8")

x = range(len(show))
axes[1].bar([i - 0.2 for i in x], [p.output_mse_int4 for p in show], width=0.4, label="output MSE @ int4", color="#e74c3c")
axes[1].bar([i + 0.2 for i in x], [p.output_mse_int8 for p in show], width=0.4, label="output MSE @ int8", color="#3498db")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1].set_ylabel("Output MSE")
axes[1].set_title("Same layer — int8 usually hurts less than int4")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Stage 4d demo: SpinQuant + ConvRot vs RTN (needs model + captures) ──
_spin_cands = [(n, m) for n, m in iter_linear_modules(model, 8) if m.weight.numel() >= MIN_PARAMS_TO_QUANT]
if _spin_cands and captures:
    _sn, _sl = _spin_cands[0]
    _xin = captures.get(_sn, torch.empty(0))
    if _xin.numel() > 0:
        _rtn_q, _rtn_s = symmetric_quantize_per_channel(_sl.weight.data, 4)
        _rtn_w = symmetric_dequant_per_channel(_rtn_q, _rtn_s)
        with torch.no_grad():
            _x = _xin[:512].to(DEVICE).float()
            _y_fp = F.linear(_x, _sl.weight.float(), _sl.bias)
            _y_rtn = F.linear(_x, _rtn_w.to(DEVICE), _sl.bias)
            mse_rtn = (_y_fp - _y_rtn).pow(2).mean().item()

        _sq = SpinQuantQuantizer(_sl, n_bits=4)
        _h = _sl.register_forward_hook(lambda m, inp, out, q=_sq: q.add_batch(inp[0].detach()))
        run_calibration(model, processor, image, PROMPT, 2)
        _h.remove()
        _ql = QuantizedLinear(_sl.in_features, _sl.out_features, _sq.quantize()).to(DEVICE)
        mse_spin = layer_output_mse(_sl, _ql, _xin)

        _cr = ConvRotQuantizer(_sl, n_bits=4)
        _st_cr = _cr.quantize()
        _ql_cr = QuantizedLinear(_sl.in_features, _sl.out_features, _st_cr).to(DEVICE)
        mse_convrot = layer_output_mse(_sl, _ql_cr, _xin)

        print(f"SpinQuant demo — {_sn}  shape={tuple(_sl.weight.shape)}")
        print(f"  RTN output MSE      : {mse_rtn:.4e}")
        print(f"  SpinQuant output MSE: {mse_spin:.4e}")
        print(f"  ConvRot output MSE  : {mse_convrot:.4e}  (group={CONVROT_GROUP_SIZE})")
        print(f"  SpinQuant improvement : {(1 - mse_spin/max(mse_rtn,1e-12))*100:.1f}% vs RTN")
        print(f"  ConvRot improvement   : {(1 - mse_convrot/max(mse_rtn,1e-12))*100:.1f}% vs RTN")
    else:
        print("SpinQuant demo skipped — no calibration inputs for demo layer.")
else:
    print("SpinQuant demo skipped — run sensitivity capture cell first.")

---
## Stage 5d — Mixed precision plan (int4 / int8 / fp16)

Rank layers by sensitivity score $S_\ell$. Partition by percentiles:

| Precision | When to use | Rule in this notebook |
|-----------|-------------|----------------------|
| **fp16** | Sensitive layers, heads, embeddings | Top `FP16_SENSITIVE_PCT`% + protected patterns |
| **int8** | Medium sensitivity — good balance | Next `INT8_MID_PCT`% |
| **int4** | Large, stable layers — max compression | Everything else |

**Storage estimate** for layer $\ell$ with bits $b_\ell$:

$$
\text{bytes}_\ell = \left\lceil \frac{O_\ell I_\ell \cdot b_\ell}{8} \right\rceil + O_\ell \cdot \text{sizeof(scale)}
$$

**Total model size** under plan $\pi$:

$$
|\theta_{\text{quant}}| = \sum_{\ell: b_\ell < 16} \text{bytes}_\ell + \sum_{\ell: b_\ell = 16} 2 O_\ell I_\ell
$$

**Proof (optimality of mixed plan):** assigning $b_\ell=4$ to lowest-$S_\ell$ layers minimizes $\sum \text{bytes}_\ell$ subject to per-layer error tolerance — greedy by sensitivity rank is the discrete analogue of water-filling.

Tweak `FP16_SENSITIVE_PCT` / `INT8_MID_PCT` if OCR quality drops. Set **`MIXED_PRECISION = False`** for uniform `BITS` (simpler, riskier).

In [ ]:
if MIXED_PRECISION:
    profiles = assign_mixed_precision(profiles)
else:
    for p in profiles:
        if p.protected:
            p.bits, p.note = "fp16", "protected"
        else:
            p.bits, p.note = f"int{BITS}", f"uniform int{BITS} (mixed off)"

from collections import Counter
bit_counts = Counter(p.bits for p in profiles)
param_by_bits = {}
for p in profiles:
    param_by_bits[p.bits] = param_by_bits.get(p.bits, 0) + p.num_params

print("Mixed precision plan")
print("=" * 70)
print(f"{'Bits':<8} {'Layers':>8} {'Params':>12} {'% of Linear':>12}")
print("-" * 70)
for bits in ["fp16", "int8", "int4"]:
    n_layers = bit_counts.get(bits, 0)
    n_params = param_by_bits.get(bits, 0)
    pct = 100.0 * n_params / max(total_linear_params, 1)
    print(f"{bits:<8} {n_layers:>8} {n_params:>12,} {pct:>11.1f}%")

print(f"\n{'Layer':<45} {'Bits':<6} {'Sens':>9}  Reason")
print("-" * 90)
for p in profiles[:20]:
    short = p.name if len(p.name) <= 44 else "…" + p.name[-43:]
    print(f"{short:<45} {p.bits:<6} {p.sensitivity:>9.2e}  {p.note}")

colors = {"fp16": "#e74c3c", "int8": "#f39c12", "int4": "#27ae60"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].pie(
    [param_by_bits.get(b, 0) for b in ["int4", "int8", "fp16"]],
    labels=["int4", "int8", "fp16"],
    colors=[colors[b] for b in ["int4", "int8", "fp16"]],
    autopct="%1.0f%%", startangle=90,
)
axes[0].set_title("Param share by precision")
show = profiles[:min(18, len(profiles))]
bar_colors = [colors.get(p.bits, "gray") for p in show]
axes[1].barh([p.name.split(".")[-1] for p in show][::-1], [p.sensitivity for p in show][::-1], color=bar_colors[::-1])
axes[1].set_xlabel("Sensitivity")
axes[1].set_title("Red=fp16  Orange=int8  Green=int4")
plt.tight_layout(); plt.show()

### Quick decision guide

Decision tree (per Linear layer $\ell$):

```
For each Linear layer:
│
├─ Name matches lm_head / embed / vision_projection?  → fp16 (never quantize)
├─ Params < MIN_PARAMS_TO_QUANT?                    → fp16 (not worth it)
├─ Output MSE @ int4 in top 15%?                    → fp16 (too sensitive)
├─ Output MSE @ int4 in next 35%?                   → int8  (safe middle ground)
└─ Everything else                                   → int4  (max savings)
```

**Formal rule:** assign $b_\ell \in \{4,8,16\}$ minimizing expected size subject to $\text{MSE}_{\text{out},\ell} \le \tau_{b_\ell}$.

**If OCR gets worse after quant:** increase `FP16_SENSITIVE_PCT`, try **`QUANT_METHOD = "convrot"`** (group RHT), **`spinquant`** (learned rotations), or **`smoothquant`** (activation outliers).

---
## Stage 6 — Apply the plan and swap layers

For each layer $\ell$ with $b_\ell \in \{4,8\}$, compute $\hat{\mathbf{W}}_\ell$ via chosen algorithm. Layers with $b_\ell = 16$ unchanged.

### Shape invariance

Replacement $\Phi: \text{Linear}(O,I) \to \text{QuantizedLinear}(O,I)$ satisfies:

$$
\forall \mathbf{x} \in \mathbb{R}^{I}: \|\Phi(L)(\mathbf{x}) - L(\mathbf{x})\|_2 \approx 0 \text{ (calibration)}
$$

and **dimensions preserved:** $\mathbf{y} \in \mathbb{R}^O$ unchanged ⇒ no graph surgery beyond module swap.

**Proof (composability):** if $\mathbf{h}_{\ell+1} = \sigma(\text{QuantLinear}_\ell(\mathbf{h}_\ell))$ matches fp output on calibration set, downstream layers see identical input distribution to fp model (first-order condition for end-to-end OCR preservation).

In [ ]:
to_quant = [p for p in profiles if p.bits in ("int4", "int8")]
to_keep = [p for p in profiles if p.bits == "fp16"]
print(f"Plan: quantize {len(to_quant)} layers  ·  keep {len(to_keep)} at fp16  ·  method={QUANT_METHOD.upper()}")
for p in to_quant[:8]:
    print(f"  {p.bits:>4}  {p.name}")
if len(to_quant) > 8:
    print(f"  ... +{len(to_quant)-8} more")

t0 = time.time()
stats = apply_quant_plan(model, profiles, QUANT_METHOD, captures)
elapsed = time.time() - t0

if stats:
    avg_mse = sum(s["weight_mse"] for s in stats) / len(stats)
    total_fp = sum(s["fp_bytes"] for s in stats)
    total_q = sum(s["q_bytes"] for s in stats)
    print(f"\nFinished in {elapsed:.1f}s")
    print(f"Average weight MSE on quant layers: {avg_mse:.2e}")
    print(f"Quantized storage: {total_q/1024:.1f} KB  vs  {total_fp/1024:.1f} KB fp16 before")
    print(f"Compression on quant layers: {total_fp/max(total_q,1):.2f}×")
else:
    print("No layers were quantized.")

In [ ]:
# Weight error on layers we actually quantized
if stats:
    colors = {"int4": "#27ae60", "int8": "#f39c12"}
    names = [s["layer"].split(".")[-1] for s in stats]
    mses = [s["weight_mse"] for s in stats]
    bar_c = [colors.get(s["bits"], "steelblue") for s in stats]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(mses)), mses, color=bar_c)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Weight MSE")
    ax.set_title(f"{QUANT_METHOD.upper()} — error per quant layer (green=int4, orange=int8)")
    plt.tight_layout(); plt.show()

---
## Stage 7 — Does OCR still work?

Quantization is approximate: $\hat{\mathbf{W}} \neq \mathbf{W}$. We verify **downstream task quality**, not weight fidelity.

### IoU acceptance criterion

For boxes $B = [x_1,y_1,x_2,y_2]$ and $\hat{B}$ from quant model:

$$
\text{IoU}(B, \hat{B}) = \frac{|B \cap \hat{B}|}{|B \cup \hat{B}|}
$$

**Success:** median IoU $\ge 0.85$ and line count $|L_{\text{quant}} - L_{\text{fp}}| \le 2$ on calibration page.

**Proof (why IoU not pixel MSE):** OCR cares about discrete structures (lines, cells); IoU is the natural metric for set overlap of rectangles. Weight MSE can be low while IoU collapses if errors align with high-gradient activation directions (AWQ/GPTQ target this, naive RTN does not).

In [ ]:
def run_florence_detect(image, processor, model, device, max_new_tokens=512):
    prompt = "<OCR_WITH_REGION>"
    padded, px, py, ow, oh = pad_info(image)
    inputs = processor(text=prompt, images=padded, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype=next(model.parameters()).dtype)
    t0 = time.time()
    with torch.no_grad():
        gen = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=max_new_tokens,
            num_beams=1,
            use_cache=False,
        )
    elapsed = time.time() - t0
    raw = processor.batch_decode(gen, skip_special_tokens=False)[0]
    side = max(ow, oh)
    parsed = processor.post_process_generation(raw, task=prompt, image_size=(side, side))
    region = parsed.get(prompt, {})
    lines = []
    for quad, label in zip(region.get("quad_boxes", []), region.get("labels", [])):
        xs, ys = quad[0::2], quad[1::2]
        bbox = [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
        x1 = max(0, min(ow, bbox[0] - px)); y1 = max(0, min(oh, bbox[1] - py))
        x2 = max(0, min(ow, bbox[2] - px)); y2 = max(0, min(oh, bbox[3] - py))
        if x2 > x1 and y2 > y1:
            text = re.sub(r"</?[a-zA-Z_][^>]*>", "", str(label)).strip()
            lines.append({"text": text, "bbox": [x1, y1, x2, y2]})
    return lines, elapsed


lines, infer_s = run_florence_detect(image, processor, model, DEVICE)
print(f"Took {infer_s:.2f}s — found {len(lines)} text lines")
for line in lines[:8]:
    print(f"  • {line['text'][:70]}")
if len(lines) > 8:
    print(f"  ... +{len(lines)-8} more lines")

vis = image.copy(); draw = ImageDraw.Draw(vis)
for line in lines:
    b = line["bbox"]
    draw.rectangle(b, outline="lime", width=2)
plt.figure(figsize=(8, 6)); plt.imshow(vis)
plt.title(f"Mixed {QUANT_METHOD.upper()} — detect overlay"); plt.axis("off"); plt.show()

---
## Stage 8 — Compare all five quant methods on full OCR

Controlled experiment with fresh $\theta^{(m)}$ per method:

$$
m \in \{\text{gptq, awq, smoothquant, spinquant, convrot}\}
$$

| Method | Core idea |
|--------|-----------|
| **GPTQ** | Hessian-aware column quant |
| **AWQ** | Activation-aware scale search |
| **SmoothQuant** | Outlier migration $s_j$ |
| **SpinQuant** | Learned Givens $\mathbf{R}$ |
| **ConvRot** | Group-wise RHT, $O(K)$ |

Controlled by **`RUN_ALL_METHODS_OCR`** (default `True`).

In [ ]:
def load_fresh_model():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, torch_dtype=dtype, attn_implementation="eager",
    ).to(DEVICE)
    m.eval()
    return m


def run_quant_ocr(method: str):
    """Fresh model → apply mixed plan → detect OCR."""
    m = load_fresh_model()
    t0 = time.time()
    apply_quant_plan(m, profiles, method, captures)
    quant_s = time.time() - t0
    lines, infer_s = run_florence_detect(image, processor, m, DEVICE)
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {"method": method, "lines": len(lines), "quant_s": quant_s,
            "infer_s": infer_s, "sample": [l["text"][:40] for l in lines[:3]]}


METHODS = ["gptq", "awq", "smoothquant", "spinquant", "convrot"]
METHOD_COLORS = {
    "gptq": "#4C72B0", "awq": "#55A868", "smoothquant": "#C44E52",
    "spinquant": "#8172B3", "convrot": "#CCB974",
}

if RUN_ALL_METHODS_OCR:
    print("Running full OCR pipeline for GPTQ, AWQ, SmoothQuant, SpinQuant, ConvRot...\n")
    ocr_results = []
    for method in METHODS:
        print(f"--- {method.upper()} ---")
        r = run_quant_ocr(method)
        ocr_results.append(r)
        print(f"  quant {r['quant_s']:.1f}s  infer {r['infer_s']:.1f}s  lines={r['lines']}")
        for t in r["sample"]:
            print(f"    • {t}")
        print()

    print(f"{'Method':<14} {'Lines':>6} {'Quant(s)':>10} {'Infer(s)':>10}")
    print("-" * 44)
    for r in ocr_results:
        print(f"{r['method']:<14} {r['lines']:>6} {r['quant_s']:>10.1f} {r['infer_s']:>10.1f}")

    fig, ax = plt.subplots(figsize=(7, 3))
    colors = METHOD_COLORS
    ax.bar([r["method"] for r in ocr_results], [r["lines"] for r in ocr_results],
           color=[colors[r["method"]] for r in ocr_results])
    ax.set_ylabel("Detected lines"); ax.set_title("OCR detect — five quant methods")
    plt.tight_layout(); plt.show()
else:
    print("Set RUN_ALL_METHODS_OCR = True to compare all five quant methods on full OCR.")


# ── Bonus: single-layer weight + output MSE for all 5 methods ──
if COMPARE_LAYER_MSE:
    print("\n" + "=" * 60)
    print("Single-layer MSE compare (weight + output, all 5 methods)")
    model_cmp = load_fresh_model()
    name0, layer0 = iter_linear_modules(model_cmp, 1)[0]
    cap0 = {}
    h = layer0.register_forward_hook(
        lambda m, inp, out: cap0.setdefault("x", []).append(inp[0].detach().reshape(-1, inp[0].shape[-1]).cpu())
    )
    run_calibration(model_cmp, processor, image, PROMPT, MAX_CALIB_BATCHES)
    h.remove()
    inp0 = torch.cat(cap0["x"], dim=0) if cap0.get("x") else torch.empty(0)
    print(f"Layer: {name0}\n")
    print(f"{'Method':<14} {'Weight MSE':>12} {'Output MSE':>12}")
    print("-" * 40)
    layer_results = []
    for method in METHODS:
        q = build_quantizer(method, layer0)
        h2 = layer0.register_forward_hook(lambda m, inp, out, q=q: q.add_batch(inp[0].detach()))
        run_calibration(model_cmp, processor, image, PROMPT, 2)
        h2.remove()
        ql = QuantizedLinear(layer0.in_features, layer0.out_features, q.quantize()).to(DEVICE)
        wm = layer_weight_mse(layer0, ql)
        om = layer_output_mse(layer0, ql, inp0)
        layer_results.append({"method": method, "weight_mse": wm, "output_mse": om})
        print(f"{method:<14} {wm:>12.2e} {om:>12.2e}")
    del model_cmp
    fig, ax = plt.subplots(figsize=(7, 3))
    x = range(len(METHODS))
    ax.bar([i - 0.2 for i in x], [r["weight_mse"] for r in layer_results], width=0.4, label="weight MSE")
    ax.bar([i + 0.2 for i in x], [r["output_mse"] for r in layer_results], width=0.4, label="output MSE")
    ax.set_xticks(list(x)); ax.set_xticklabels([r["method"] for r in layer_results])
    ax.legend(); ax.set_title("Per-method error on first Linear layer"); plt.tight_layout(); plt.show()

---
## What we did

| Step | Summary |
|------|---------|
| **Inventory** | Listed all Linear layers — size, % of params, protect flags |
| **Sensitivity** | Measured output MSE @ int4/int8 on real OCR calibration data |
| **Mixed plan** | Assigned fp16 / int8 / int4 per layer based on sensitivity rank |
| **SpinQuant demo** | Learned Givens rotations — RTN vs SpinQuant vs ConvRot output MSE |
| **ConvRot demo** | Group-wise RHT — regular Hadamard, O(K) complexity |
| **GPTQ / AWQ / SmoothQuant / SpinQuant / ConvRot** | Applied via mixed-precision plan (Stage 6) |
| **5-method OCR compare** | Stage 8 — fresh model per method, detect line counts |
| **Apply** | Swapped quant layers in; left sensitive layers at fp16 |
| **Verify** | Stage 7 — detect with chosen `QUANT_METHOD` |

**Decision cheat sheet:**
- High output MSE @ int4 → keep **fp16** or use **int8**
- Large layer + low sensitivity → **int4** (best compression)
- `lm_head` / embed / vision projection → always **fp16**
- OCR quality bad? Try **`convrot`** (group RHT), **`spinquant`**, or **`smoothquant`**

No quant libraries — just PyTorch. Tune `QUANT_METHOD`, `CONVROT_GROUP_SIZE`, `SPINQUANT_N_GIVENS`, or re-run Stage 8.

**Next:** [03 — Mobile Deployment](03_ocr_pipeline_mobile.ipynb) turns fake quant weights into packed binaries and app specs.